## 335 - Accessing and Plotting NEXRAD Level 2 Data

[Youtube](https://www.youtube.com/watch?v=C4EOzJste1Y)

In [1]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime, timedelta
from pathlib import Path
from metpy.calc import azimuth_range_to_lat_lon
from metpy.io import Level2File
from metpy.plots import USCOUNTIES, add_timestamp, colortables
from metpy.remote import NEXRADLevel2Archive
from metpy.units import units

In [2]:
site = 'KTLX'
start = datetime(2023, 4, 20)
end = start + timedelta(hours=2)

products = list(NEXRADLevel2Archive().get_range(site, start, end))
print(f'Found {len(products)} products')

Found 18 products


In [3]:
outdir = Path('level2')
outdir.mkdir(exist_ok=True)

In [4]:
extent = [-97.95, -97.0, 34.85, 35.65]
norm, cmap = colortables.get_with_steps('NWSStormClearReflectivity', -20, 0.5)

In [5]:
for i, prod in enumerate(products):
    print(f'Processing {i+1}/{len(products)}: {prod.name}')

    f = prod.access()
    sweep = 0
    rays = f.sweeps[sweep]

    ref_hdr = rays[0][4][b'REF'][0]
    data = np.array([ray[4][b'REF'][1] for ray in rays])
    data = np.ma.masked_invalid(data)

    az = np.array([ray[0].az_angle for ray in rays])

    diff = np.diff(az)
    crossed = diff < -180
    diff[crossed] +=360
    avg_spacing = diff.mean()

    az = (az[:-1] + az[1:]) / 2
    az[crossed] += 180
    az = np.concatenate(([az[0] - avg_spacing], az, [az[-1] + avg_spacing])) * units.degree
    rng = ((np.arange(ref_hdr.num_gates+1) - 0.5) * ref_hdr.gate_width + ref_hdr.first_gate) * units.kilometer

    lon0 = rays[0][1].lon
    lat0 = rays[0][1].lat

    lon, lat = azimuth_range_to_lat_lon(az, rng, lon0, lat0)

    fig = plt.figure(figsize=(9,8))

    ax = plt.axes(projection=ccrs.LambertConformal(central_longitude=lon0, central_latitude=lat0))

    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(USCOUNTIES.with_scale('5m'), linewidth=0.4)

    mesh = ax.pcolormesh(lon, lat, data, cmap=cmap, norm=norm, transform=ccrs.PlateCarree())

    cbar = plt.colorbar(mesh, ax=ax, pad=0.02, shrink=0.9)
    cbar.set_label('Reflectivity (dBZ)')

    add_timestamp(ax, f.dt, y=0.02, high_contrast=True, pretext='')

    outfile = outdir / f'frame_{i:03d}.png'
    plt.savefig(outfile, bbox_inches='tight')
    plt.close(fig)


Processing 1/18: KTLX20230420_000345_V06
Processing 2/18: KTLX20230420_001002_V06
Processing 3/18: KTLX20230420_001609_V06
Processing 4/18: KTLX20230420_002236_V06
Processing 5/18: KTLX20230420_002926_V06
Processing 6/18: KTLX20230420_003544_V06
Processing 7/18: KTLX20230420_004221_V06
Processing 8/18: KTLX20230420_004858_V06
Processing 9/18: KTLX20230420_005549_V06
Processing 10/18: KTLX20230420_010216_V06
Processing 11/18: KTLX20230420_010844_V06
Processing 12/18: KTLX20230420_011535_V06
Processing 13/18: KTLX20230420_012226_V06
Processing 14/18: KTLX20230420_012903_V06
Processing 15/18: KTLX20230420_013540_V06
Processing 16/18: KTLX20230420_014231_V06
Processing 17/18: KTLX20230420_014922_V06
Processing 18/18: KTLX20230420_015559_V06
